# ДЗ 2 -- ChannelTokenFormer
> Сорочан Дмитрий, ПАДИИ3

## 00. Данные и подготовка

Беру UCI HAR -- 9 инерциальных каналов, 128 отсчетов, 6 активностей.
Официальный split: 7352 train и 2947 test окон, разные люди.
Исходная частота -- 50 Гц, длительность окна -- 2.56 с.

[Корпус UCI](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones).
Использую raw inertial signals, а не готовые 561 признака.
Ноутбуки запускаю по порядку `00 -> 01 -> 02 -> 03`, каждый в чистом kernel.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
for folder in ["data", "figures", "results"]:
    (ROOT / folder).mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

import io
import zipfile
from urllib.request import urlretrieve
from scipy.signal import resample_poly
from sklearn.model_selection import GroupKFold

SEED = 42
SIGNALS = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]
CLASS_NAMES = [
    "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
    "SITTING", "STANDING", "LAYING",
]
STEPS = np.array([1, 2, 3, 4, 5, 2, 3, 4, 5])
FS = 50
T = 128


### Загружаю данные

Архив сохраняю локально -- при повторном запуске заново не скачиваю.
Читаю файлы прямо из zip. Старые результаты и подготовленные данные не использую.


In [2]:
url = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
archive_path = ROOT / "data/uci_har.zip"
if not archive_path.exists():
    download_path = archive_path.with_suffix(".download")
    urlretrieve(url, download_path)
    download_path.replace(archive_path)


def load_split(archive, split):
    names = archive.namelist()
    def read(suffix):
        name = next(name for name in names if name.endswith(suffix))
        return np.loadtxt(io.BytesIO(archive.read(name)))
    X = np.stack([
        read(f"{split}/Inertial Signals/{name}_{split}.txt") for name in SIGNALS
    ], axis=1).astype("float32")
    y = read(f"{split}/y_{split}.txt").astype("int64") - 1
    subjects = read(f"{split}/subject_{split}.txt").astype("int64")
    return X, y, subjects


with zipfile.ZipFile(archive_path) as outer:
    # В официальном архиве может лежать еще один zip с самим корпусом.
    inner = next((n for n in outer.namelist() if n.endswith("UCI HAR Dataset.zip")), None)
    if inner is not None:
        with zipfile.ZipFile(io.BytesIO(outer.read(inner))) as archive:
            X_train_raw, y_train, subjects_train = load_split(archive, "train")
            X_test_raw, y_test, subjects_test = load_split(archive, "test")
    else:
        X_train_raw, y_train, subjects_train = load_split(outer, "train")
        X_test_raw, y_test, subjects_test = load_split(outer, "test")

print("train:", X_train_raw.shape, "test:", X_test_raw.shape)
print("общие subjects:", np.intersect1d(subjects_train, subjects_test))


train: (7352, 9, 128) test: (2947, 9, 128)
общие subjects: []


### Разные частоты

Перед прореживанием применяю anti-aliasing FIR из `resample_poly` внутри каждого окна.
Это имитация более медленных датчиков. Фильтрация идет до отказов датчика:
потом скрытые значения не используются ни для заполнения, ни для нормализации.
Края фильтра дополняю продолжением линии; между окнами и людьми ничего не смешиваю.

На общей сетке 50 Гц храню только реально полученные отсчеты.
`scheduled` отмечает расписание измерений; остальные позиции -- NaN, а не нули.


In [3]:
def change_rates(X):
    result = np.full_like(X, np.nan)
    for channel, step in enumerate(STEPS):
        sampled = X[:, channel] if step == 1 else resample_poly(
            X[:, channel], up=1, down=int(step), axis=-1, padtype="line"
        )
        result[:, channel, ::step] = sampled
    return result


scheduled = np.arange(T)[None, :] % STEPS[:, None] == 0
X_train_rates = change_rates(X_train_raw)
X_test_rates = change_rates(X_test_raw)
rate_table = pd.DataFrame({
    "channel": SIGNALS, "step": STEPS, "sampling_hz": FS / STEPS,
    "period_ms": 1000 * STEPS / FS, "samples_per_window": scheduled.sum(axis=1),
})
rate_table.to_csv(ROOT / "results/00_sampling.csv", index=False)
rate_table


,channel,step,sampling_hz,period_ms,samples_per_window
0,body_acc_x,1,50.000000,20.0,128
1,body_acc_y,2,25.000000,40.0,64
2,body_acc_z,3,16.666667,60.0,43
3,body_gyro_x,4,12.500000,80.0,32
4,body_gyro_y,5,10.000000,100.0,26
5,body_gyro_z,2,25.000000,40.0,64
6,total_acc_x,3,16.666667,60.0,43
7,total_acc_y,4,12.500000,80.0,32
8,total_acc_z,5,10.000000,100.0,26


### Периоды пропусков

В каждом окне и канале скрываю один непрерывный интервал.
Начала выбираю независимо от сигналов, меток и людей. Train и test имеют разные seed.
Основной режим -- 32 шага (0.64 с, 25% длительности окна).
Дополнительные test-режимы -- без отказа и 64 шага (1.28 с, 50%).
Для test расширяю тот же интервал -- более длинный пропуск включает короткий.
У края сдвигаю интервал внутрь окна, сохраняя заданную длительность.

Проценты относятся к времени отказа, а не к общей доле NaN:
в нее дополнительно входят позиции вне расписания измерений.
Это контролируемое повреждение готовых окон, не реконструкция непрерывной записи.


In [4]:
def gap_starts(n, seed):
    return np.random.default_rng(seed).integers(0, T - 32 + 1, size=(n, 9, 1))


def add_gaps(X, starts, length):
    # Расширяю основной интервал в обе стороны; у края сдвигаю внутрь окна.
    starts = np.clip(starts + (32 - length) // 2, 0, T - length)
    time_grid = np.arange(T)[None, None, :]
    block = (time_grid >= starts) & (time_grid < starts + length)
    return np.where(block, np.nan, X).astype("float32"), block


train_starts = gap_starts(len(X_train_raw), SEED)
test_starts = gap_starts(len(X_test_raw), SEED + 1)
X_train, train_blocks = add_gaps(X_train_rates, train_starts, 32)
X_test_0, _ = add_gaps(X_test_rates, test_starts, 0)
X_test_25, _ = add_gaps(X_test_rates, test_starts, 32)
X_test_50, _ = add_gaps(X_test_rates, test_starts, 64)

# Окна перекрываются -- поэтому validation выделяю целыми людьми.
fold_id = np.zeros(len(y_train), dtype="int64")
for fold, (_, val_idx) in enumerate(GroupKFold(3).split(X_train, y_train, subjects_train)):
    fold_id[val_idx] = fold

split_rows = []
for fold in range(3):
    fit_subjects = np.unique(subjects_train[fold_id != fold])
    val_subjects = np.unique(subjects_train[fold_id == fold])
    split_rows.append({
        "fold": fold + 1, "fit_subjects": fit_subjects.tolist(),
        "val_subjects": val_subjects.tolist(),
        "overlap": np.intersect1d(fit_subjects, val_subjects).size,
    })
display(pd.DataFrame(split_rows))


,fold,fit_subjects,val_subjects,overlap
0,1,"[5, 6, 7, 14, 15, 16, 19, 21, 22, 23, 26, 28, ...","[1, 3, 8, 11, 17, 25, 27]",0
1,2,"[1, 3, 5, 6, 8, 11, 17, 19, 22, 23, 25, 26, 27...","[7, 14, 15, 16, 21, 28, 29]",0
2,3,"[1, 3, 7, 8, 11, 14, 15, 16, 17, 21, 25, 27, 2...","[5, 6, 19, 22, 23, 26, 30]",0


### Сохраняю

Модели читают только `uci_har_missing.npz` -- в нем нет скрытых исходных значений.
Чистые train-сигналы сохраняю отдельно только для иллюстраций EDA.
Все обучаемые статистики считаются позже, внутри train fold.


In [5]:
np.savez_compressed(
    ROOT / "data/uci_har_missing.npz",
    X_train=X_train, y_train=y_train, subjects_train=subjects_train,
    X_test_0=X_test_0, X_test_25=X_test_25, X_test_50=X_test_50,
    y_test=y_test, subjects_test=subjects_test, fold_id=fold_id,
    scheduled=scheduled, steps=STEPS, train_starts=train_starts, test_starts=test_starts,
)
np.savez_compressed(ROOT / "data/uci_har_eda.npz",
                    X_train_raw=X_train_raw, X_train_rates=X_train_rates, train_blocks=train_blocks)
metadata = {
    "seed": SEED, "source": url, "channels": SIGNALS, "classes": CLASS_NAMES,
    "original_hz": FS, "sequence_length": T, "steps": STEPS.tolist(),
    "train_gap_steps": 32, "test_gap_steps": [0, 32, 64],
    "split": split_rows, "test_subjects": np.unique(subjects_test).tolist(),
}
(ROOT / "data/metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
print("saved: data/uci_har_missing.npz")


saved: data/uci_har_missing.npz
